In [4]:
%pip install brian2 brian2hears librosa numpy scipy matplotlib
%pip install setuptools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import logging
logging.getLogger('brian2').setLevel(logging.ERROR)

import brian2
brian2.prefs.codegen.target = 'numpy'

from brian2 import Hz, kHz
from brian2hears import Sound, erbspace, Gammatone, Filterbank
from scipy import signal
import librosa
import numpy as np
import matplotlib.pyplot as plt
from math import gcd 

class EnvelopeFromGammatoneFilterbank(Filterbank):
    """Converts the output of a GammatoneFilterbank to an envelope."""

    def __init__(self, source):
        super().__init__(source)
        self.nchannels = 1

    def buffer_apply(self, input_):
        # 6. take absolute value of the input_
        abs_input = np.abs(input_)
        # 7. power-law compression (exponent 0.6)
        compressed = abs_input ** 0.6
        # 8. linearly combine by summing the subbands
        envelope = np.sum(compressed, axis=1, keepdims=True) 
        return envelope

def process_audio_file(wav_filename, mode='cnn'):
    if mode == 'linear':
        target_sr = 20
        lowcut = 1.0
        highcut = 9.0
    elif mode == 'cnn':
        target_sr = 64
        lowcut = 1.0
        highcut = 32.0
    else:
        raise ValueError("Kies 'linear' of 'cnn' als mode.")
    # 1. LOAD THE AUDIO FILE
    audio_data, sr = librosa.load(wav_filename, sr=None, mono=True)

    # 2. convert the audio file to a brian sound object
    sound = Sound(audio_data.reshape(-1, 1), samplerate=sr*Hz) #audio in 1 kolom (kanaal) door reshape vooor brian


    # 3. 28 center frequencies between 50 Hz and 5 kHz
    cf = erbspace(50*Hz, 5*kHz, 28)

    # 4. create the gammatone filterbank
    gammatone_filterbank = Gammatone(sound, cf)

    # 5. process envelope
    envelope_calcuation = EnvelopeFromGammatoneFilterbank(gammatone_filterbank)
    envelope = envelope_calcuation.process()
    envelope = envelope.flatten() 

    # 6. Bandpass filter instellen (1-32 Hz)
    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=sr, output='sos')
    envelope_filtered = signal.sosfiltfilt(sos, envelope)

    # 7. Downsample de resulterende signalen naar 64Hz
    g = gcd(int(sr), target_sr)
    envelope_downsampled = signal.resample_poly(envelope_filtered, target_sr // g, int(sr) // g)
    
    return envelope_downsampled

test_audio = "data/stimuli/stimuli_1/audiobook_1_1.wav"


env = process_audio_file(test_audio)
print(f"Lengte van envelope: {len(env)} samples.")
envelope_LR = process_audio_file(test_audio, mode='linear')    
# Visuele check: plot de eerste 10 seconden (10 * 64 Hz = 640 samples) (voor cnn 64 hz, voor lineair 20)
plt.figure(figsize=(10, 4))
plt.plot(env[:640])
plt.title("Gefilterde en Downsampled Envelope")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.show()
    



KeyboardInterrupt: 

In [10]:
def process_eeg_file(npz_filename, mode='cnn'):
    print(f"Bezig met inladen van EEG: {npz_filename}...")
    
    # --- Stap 1: Load the EEG data ---
    data = np.load(npz_filename)
    eeg_data = data['eeg']  # Vorm: (116736, 64)
    fs = int(data['fs'])    # 128 Hz
    
    # Handig voor straks: haal direct de paden van de bijbehorende audio op!
    attended_wav = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])

    # --- Instellingen op basis van mode (Exact als bij audio) ---
    if mode == 'linear':
        target_sr = 20
        lowcut = 1.0
        highcut = 9.0
    elif mode == 'cnn':
        target_sr = 64
        lowcut = 1.0
        highcut = 32.0
    else:
        raise ValueError("Kies 'linear' of 'cnn' als mode.")

    # --- Stap 2: Bandpass filter (Exact dezelfde bewerking als audio) ---
    # Let op: fs is hier de fs van de EEG (128 Hz)
    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    
    # We voegen axis=0 toe, zodat hij filtert over de tijd, niet over de 64 kanalen heen
    eeg_filtered = signal.sosfiltfilt(sos, eeg_data, axis=0)

    # --- Stap 3: Downsample de EEG signals ---
    g = gcd(fs, target_sr)
    # Ook hier axis=0 toevoegen
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)
    
    print(f"Succes! Vorm veranderd van {eeg_data.shape} naar {eeg_downsampled.shape}")
    
    return eeg_downsampled, attended_wav, unattended_wav

# --- TEST DE FUNCTIE ---
eeg_pad = "data/64 Channel Biosemi unprocessed data - train/sub-01-10/sub-001/sub-001_-_audiobook_1.npz"
eeg_processed, att_audio, unatt_audio = process_eeg_file(eeg_pad, mode='cnn')

print(f"Attended audio file volgens de EEG: {att_audio}")
print(f"Unattended audio file volgens de EEG: {unatt_audio}")

Bezig met inladen van EEG: data/64 Channel Biosemi unprocessed data - train/sub-01-10/sub-001/sub-001_-_audiobook_1.npz...
Succes! Vorm veranderd van (116736, 64) naar (58368, 64)
Attended audio file volgens de EEG: audiobook_1.wav
Unattended audio file volgens de EEG: audiobook_5_2.wav
